# EFSM Quick Evaluation for Colab

This notebook compares base vs fine-tuned responses using one loaded PEFT model. It generates the base response by temporarily disabling the LoRA adapter, then generates the fine-tuned response with the adapter enabled.

## Cell 1 - HF Token

If you saved `HF_TOKEN` in Colab secrets, this cell reads it. Otherwise it asks you to paste it securely.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = None

if not token:
    token = getpass('Paste HF_TOKEN: ')

os.environ['HF_TOKEN'] = token
print('HF token loaded.')

## Cell 2 - Clone Latest Repo and Install

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/tasbidrahman10/empathetic-voice-llm.git'
REPO_DIR = '/content/efsm-code-fixed'

os.chdir('/content')
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
print('Repo commit:')
subprocess.run(['git', 'log', '-1', '--oneline'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
print('Requirements installed.')
print('Torch CUDA available:', __import__('torch').cuda.is_available())

## Cell 3 - Run Base vs Fine-Tuned Comparison

This uses one loaded PEFT model: adapter disabled for base response, adapter enabled for fine-tuned response.

In [ ]:
import os
import subprocess
import sys

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
alloc_conf = 'expandable_segments:True,max_split_size_mb:64,garbage_collection_threshold:0.8'
env['PYTORCH_ALLOC_CONF'] = alloc_conf
env['PYTORCH_CUDA_ALLOC_CONF'] = alloc_conf

cmd = [
    sys.executable,
    'src/eval/quick_compare.py',
    '--config', 'configs/config.yaml',
    '--output', 'results/quick_eval_results.csv',
    '--limit', '6',
    '--max-new-tokens', '80',
    '--device-strategy', 'single_gpu_4bit',
    '--compare-mode', 'single_peft',
    '--system-prompt-style', 'therapeutic',
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)

## Optional Fallback - Lower Memory

Run this only if Cell 3 OOMs. It uses CPU offload and only 3 prompts.

In [ ]:
# OPTIONAL FALLBACK ONLY
import os
import subprocess
import sys

env = os.environ.copy()
env['CUDA_VISIBLE_DEVICES'] = '0'
alloc_conf = 'expandable_segments:True,max_split_size_mb:64,garbage_collection_threshold:0.8'
env['PYTORCH_ALLOC_CONF'] = alloc_conf
env['PYTORCH_CUDA_ALLOC_CONF'] = alloc_conf

cmd = [
    sys.executable,
    'src/eval/quick_compare.py',
    '--config', 'configs/config.yaml',
    '--output', 'results/quick_eval_results.csv',
    '--limit', '3',
    '--max-new-tokens', '60',
    '--device-strategy', 'cpu_offload',
    '--compare-mode', 'single_peft',
    '--system-prompt-style', 'therapeutic',
]

print('Running fallback:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)

## Cell 4 - Display Comparison Table

In [ ]:
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 500)
df = pd.read_csv('results/quick_eval_results.csv')
display(df[['emotion', 'prompt', 'base_response', 'fine_tuned_response']])
df.to_csv('results/quick_eval_results.csv', index=False)
print('Saved results/quick_eval_results.csv')

## Scoring Guide

For each row, score base and fine-tuned responses from 1 to 5 on emotional acknowledgement, warmth, relevance, and therapeutic tone. For the supervisor demo, show the strongest 5-6 rows plus the W&B eval-loss curve.